In [1]:
import os
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
import cv2
import nibabel as nib
from torchvision import transforms
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.cuda.amp import autocast, GradScaler
import matplotlib.pyplot as plt


In [5]:
# Exemplo de uso:
# Você pode ter transformações mais complexas (aumento de dados) aqui.
# Para um início simples, apenas redimensione se suas imagens tiverem tamanhos diferentes.
data_transform = transforms.Compose([
    # transforms.Resize((256, 256)), # Se quiser redimensionar todas as imagens para um tamanho fixo
    # transforms.ToTensor(), # Se usar isso, remova o np.expand_dims e /255.0 manuais acima
])

# Ajuste os caminhos para suas pastas
img_dir = r'C:\Users\sthem\OneDrive\Documentos\GitHub\master-2025\3-Datasets\imagens-alemanha\img_dir\train' # Verifique se estes caminhos estão corretos
mask_dir = r'C:\Users\sthem\OneDrive\Documentos\GitHub\master-2025\3-Datasets\imagens-alemanha\annot_dir\train_segmentadas\train_segmentadas' # Verifique se estes caminhos estão corretos

# Certifique-se de que os diretórios existem e contêm os arquivos
if not os.path.exists(img_dir) or not os.path.exists(mask_dir):
    print("Por favor, verifique os caminhos para as pastas de imagens e máscaras.")
    print(f"Diretório de Imagens: {img_dir}")
    print(f"Diretório de Máscaras: {mask_dir}")
    # Você pode criar umas imagens de exemplo para testar:
    # np.random.rand(1088, 2048).astype(np.float32) * 255
    # cv2.imwrite(os.path.join(img_dir, 'image1.bmp'), fake_img)
    # fake_mask = np.zeros((2048, 1088)).astype(np.float32)
    # fake_mask[300:400, 500:600] = 1
    # nib.Nifti1Image(fake_mask, np.eye(4)).to_filename(os.path.join(mask_dir, 'mask1.nii.gz'))
    # exit() # Saia se os diretórios não existirem para evitar erros (recomendo descomentar em script final)

dataset = BloodCellDataset(img_dir=img_dir, mask_dir=mask_dir, transform=data_transform)
dataloader = DataLoader(dataset, batch_size=4, shuffle=True, num_workers=0) # Altere num_workers para 0 se tiver problemas no Windows

Atenção: Máscara não encontrada para a imagem 2022_03_21_11_02_58_05.bmp. Será usada uma máscara vazia.
Atenção: Máscara não encontrada para a imagem 2022_03_21_11_02_58_14.bmp. Será usada uma máscara vazia.
Atenção: Máscara não encontrada para a imagem 2022_03_21_11_02_58_22.bmp. Será usada uma máscara vazia.
Atenção: Máscara não encontrada para a imagem 2022_03_21_11_02_58_31.bmp. Será usada uma máscara vazia.
Atenção: Máscara não encontrada para a imagem 2022_03_21_11_02_58_39.bmp. Será usada uma máscara vazia.
Atenção: Máscara não encontrada para a imagem 2022_03_21_11_02_58_56.bmp. Será usada uma máscara vazia.
Atenção: Máscara não encontrada para a imagem 2022_03_21_11_02_58_64.bmp. Será usada uma máscara vazia.
Atenção: Máscara não encontrada para a imagem 2022_03_21_11_02_58_72.bmp. Será usada uma máscara vazia.
Atenção: Máscara não encontrada para a imagem 2022_03_21_11_02_58_89.bmp. Será usada uma máscara vazia.
Atenção: Máscara não encontrada para a imagem 2022_03_21_11_02_5

In [2]:

class BloodCellDataset(Dataset):
    def __init__(self, img_dir, mask_dir, transform=None):
        self.img_dir = img_dir
        self.mask_dir = mask_dir
        self.transform = transform
        self.img_filenames = sorted([f for f in os.listdir(img_dir) if f.endswith('.bmp')])
        # Não assuma que mask_filenames corresponde 1:1 ainda.
        # Vamos criar um mapeamento para lidar com as máscaras faltantes.
        
        self.data_pairs = []
        for img_name in self.img_filenames:
            # Assumindo que o nome da máscara é igual ao da imagem, mudando apenas a extensão
            base_name = os.path.splitext(img_name)[0]
            mask_name = base_name + '.nii.gz'
            mask_path = os.path.join(mask_dir, mask_name)

            if os.path.exists(mask_path):
                self.data_pairs.append((img_name, mask_name))
            else:
                # Se a máscara não existe, adicionamos um 'None' como marcador.
                # Isso significa que essa imagem não tem malária.
                self.data_pairs.append((img_name, None))
                print(f"Atenção: Máscara não encontrada para a imagem {img_name}. Será usada uma máscara vazia.")

        print(f"Total de pares (imagem, máscara): {len(self.data_pairs)}")
        print(f"Imagens sem máscara correspondente (malária): {len([p for p in self.data_pairs if p[1] is None])}")


    def __len__(self):
        return len(self.data_pairs)

    def __getitem__(self, idx):
        img_name, mask_name = self.data_pairs[idx]

        img_path = os.path.join(self.img_dir, img_name)

        # Carregar imagem .bmp em escala de cinza
        image = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
        if image is None:
            raise FileNotFoundError(f"Não foi possível carregar a imagem: {img_path}")
        image = image.astype(np.float32) / 255.0 # Normalizar para 0-1

        # Lidar com a máscara
        if mask_name is None:
            # Criar uma máscara totalmente preta (sem malária) com as dimensões da imagem
            mask = np.zeros_like(image, dtype=np.float32)
        else:
            mask_path = os.path.join(self.mask_dir, mask_name)
            nifti_mask = nib.load(mask_path)
            mask = nifti_mask.get_fdata()
            
            # Lidar com dimensões da máscara (assumindo 2D ou primeira fatia de 3D)
            if mask.ndim > 2:
                mask = mask[:, :, 0] # Pegar a primeira fatia se for 3D

            # **Importante:** Alinhar as dimensões da máscara com a imagem (sua correção anterior)
            if mask.shape == (image.shape[1], image.shape[0]): # Se (largura_img, altura_img)
                mask = np.transpose(mask)
            
            # Garantir que a máscara tem as mesmas dimensões da imagem
            if mask.shape != image.shape:
                 mask = cv2.resize(mask.astype(np.uint8), 
                                   (image.shape[1], image.shape[0]), 
                                   interpolation=cv2.INTER_NEAREST)
            
            mask = (mask > 0).astype(np.float32) # Máscara binária (0 ou 1)


        # PyTorch espera (C, H, W) para imagens e máscaras
        image = np.expand_dims(image, axis=0) # Adicionar dimensão do canal (1, H, W)
        mask = np.expand_dims(mask, axis=0)   # Adicionar dimensão do canal (1, H, W)

        image_tensor = torch.from_numpy(image)
        mask_tensor = torch.from_numpy(mask)

        if self.transform:
            # Transformações (se houver, como redimensionamento, devem ser aplicadas aqui)
            # Se usar transforms.ToTensor() aqui, remova os np.expand_dims e /255.0 manuais.
            pass 

        return image_tensor, mask_tensor

# O restante do seu código (dataloader, modelo, treinamento, avaliação) permanece o mesmo.

In [3]:

class DoubleConv(nn.Module):
    """(convolution => BN => ReLU) * 2"""
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.double_conv = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.double_conv(x)

class Down(nn.Module):
    """Downscaling with maxpool then double conv"""
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.maxpool_conv = nn.Sequential(
            nn.MaxPool2d(2),
            DoubleConv(in_channels, out_channels)
        )

    def forward(self, x):
        return self.maxpool_conv(x)

class Up(nn.Module):
    """Upscaling then double conv"""
    def __init__(self, in_channels, out_channels, bilinear=True):
        super().__init__()
        # if bilinear, use the normal convolutions to reduce the number of channels
        if bilinear:
            self.up = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)
            self.conv = DoubleConv(in_channels, out_channels)
        else:
            self.up = nn.ConvTranspose2d(in_channels, in_channels // 2, kernel_size=2, stride=2)
            self.conv = DoubleConv(in_channels, out_channels)


    def forward(self, x1, x2):
        x1 = self.up(x1)
        # Pad x1 to match the shape of x2 (due to pooling/upsampling potentially creating off-by-one differences)
        diffY = x2.size()[2] - x1.size()[2]
        diffX = x2.size()[3] - x1.size()[3]

        x1 = F.pad(x1, [diffX // 2, diffX - diffX // 2,
                        diffY // 2, diffY - diffY // 2])
        
        x = torch.cat([x2, x1], dim=1) # Concatena via canal (skip connection)
        return self.conv(x)

class OutConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super(OutConv, self).__init__()
        self.conv = nn.Conv2d(in_channels, out_channels, kernel_size=1)

    def forward(self, x):
        return self.conv(x)

class UNet(nn.Module):
    def __init__(self, n_channels, n_classes, bilinear=True):
        super(UNet, self).__init__()
        self.n_channels = n_channels
        self.n_classes = n_classes
        self.bilinear = bilinear

        self.inc = DoubleConv(n_channels, 64)
        self.down1 = Down(64, 128)
        self.down2 = Down(128, 256)
        self.down3 = Down(256, 512)
        self.down4 = Down(512, 1024)
        self.up1 = Up(1024 + 512, 512, bilinear) # 1024 do layer abaixo + 512 do skip connection
        self.up2 = Up(512 + 256, 256, bilinear)
        self.up3 = Up(256 + 128, 128, bilinear)
        self.up4 = Up(128 + 64, 64, bilinear)
        self.outc = OutConv(64, n_classes)

    def forward(self, x):
        x1 = self.inc(x)
        x2 = self.down1(x1)
        x3 = self.down2(x2)
        x4 = self.down3(x3)
        x5 = self.down4(x4)
        x = self.up1(x5, x4) # Passa x5 e x4 (skip connection)
        x = self.up2(x, x3)
        x = self.up3(x, x2)
        x = self.up4(x, x1)
        logits = self.outc(x)
        return logits

# Instanciar o modelo (1 canal de entrada para escala de cinza, 1 canal de saída para máscara binária)
model = UNet(n_channels=1, n_classes=1) # 1 canal de saída para segmentação binária

# Mover o modelo para a GPU se disponível
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)

print(f"Modelo U-Net carregado para {device}")

Modelo U-Net carregado para cuda


In [4]:


# Definição da função de perda (Binary Cross-Entropy com Logits para estabilidade numérica)
criterion = nn.BCEWithLogitsLoss()

# Otimizador
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Scaler para treinamento com precisão mista (recomendado para GPUs)
scaler = GradScaler()

num_epochs = 10 # Número de épocas de treinamento (ajuste conforme necessário)

print("Iniciando o treinamento...")

for epoch in range(num_epochs):
    model.train() # Coloca o modelo em modo de treinamento
    running_loss = 0.0
    
    for batch_idx, (images, masks) in enumerate(dataloader):
        images = images.to(device)
        masks = masks.to(device)

        # Zero os gradientes
        optimizer.zero_grad()

        # Forward pass com autocast (precisão mista)
        with autocast():
            outputs = model(images)
            loss = criterion(outputs, masks) # Máscaras devem ser float32

        # Backward pass e otimização com GradScaler
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        running_loss += loss.item() * images.size(0)

        if batch_idx % 10 == 0: # Imprimir a cada 10 batches
            print(f"Epoch {epoch+1}/{num_epochs}, Batch {batch_idx}/{len(dataloader)}, Loss: {loss.item():.4f}")

    epoch_loss = running_loss / len(dataloader.dataset)
    print(f"Epoch {epoch+1} completa. Loss Média: {epoch_loss:.4f}")

    # (Opcional) Salvar o modelo após algumas épocas
    # torch.save(model.state_dict(), f'unet_blood_cells_epoch_{epoch+1}.pth')

print("Treinamento concluído!")

C:\Users\sthem\AppData\Local\Temp\ipykernel_22204\2587273512.py:8: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()


Iniciando o treinamento...


NameError: name 'dataloader' is not defined

In [ ]:

# Carregar o modelo treinado (se você salvou)
# model.load_state_dict(torch.load('unet_blood_cells_epoch_X.pth'))
model.eval() # Coloca o modelo em modo de avaliação (desativa dropout/batchnorm para inferência)

# Pegar um batch do dataloader de teste (ou uma imagem individual)
# Para um exemplo rápido, vamos usar uma imagem do próprio dataloader de treinamento
# ou carregar uma nova imagem.
# Supondo que você tenha um `test_dataloader` similar ao `dataloader`

# Exemplo com uma única imagem para demonstração
# Carregue uma nova imagem (que não foi usada no treinamento)
test_img_path = 'caminho/para/nova_imagem_celula.bmp'
test_image_raw = cv2.imread(test_img_path, cv2.IMREAD_GRAYSCALE)
if test_image_raw is None:
    print(f"Erro: Não foi possível carregar a imagem de teste em {test_img_path}")
else:
    test_image = test_image_raw.astype(np.float32) / 255.0
    test_image_tensor = torch.from_numpy(np.expand_dims(test_image, axis=(0, 1))).to(device) # (1, 1, H, W)

    with torch.no_grad(): # Desativa o cálculo de gradientes para inferência
        # Obter a previsão da U-Net
        prediction_logits = model(test_image_tensor)
        
        # Aplicar sigmoid para obter probabilidades (se a saída não for sigmoidalizada)
        # e depois binarizar com um limiar (0.5 é comum)
        prediction_mask = torch.sigmoid(prediction_logits).cpu().numpy()
        prediction_mask_binary = (prediction_mask > 0.5).astype(np.uint8) * 255 # Para visualização

    # Visualizar a imagem original e a máscara prevista
    plt.figure(figsize=(12, 6))
    plt.subplot(1, 2, 1)
    plt.title("Imagem Original")
    plt.imshow(test_image_raw, cmap='gray')
    plt.subplot(1, 2, 2)
    plt.title("Máscara Prevista pela U-Net")
    plt.imshow(prediction_mask_binary[0, 0, :, :], cmap='gray') # Remover as dimensões do batch e canal
    plt.show()

    # Combinar com a imagem original para destacar (igual ao método anterior)
    image_display_unet = cv2.cvtColor(test_image_raw, cv2.COLOR_GRAY2BGR)
    red_color = [0, 0, 255]
    # Lembre-se de redimensionar a máscara prevista se o modelo redimensionou internamente.
    # Se você usou transforms.Resize no dataloader, a previsão estará nesse tamanho.
    # Aqui, assumimos que o modelo outputa o mesmo tamanho de entrada (U-Net padrão).
    mask_for_overlay = prediction_mask_binary[0, 0, :, :] # Pegar a máscara 2D
    image_display_unet[mask_for_overlay > 0] = red_color

    plt.figure(figsize=(10, 8))
    plt.title("Células Infectadas (Previstas por U-Net) Destacadas")
    plt.imshow(cv2.cvtColor(image_display_unet, cv2.COLOR_BGR2RGB))
    plt.show()